# 10 — Scheduling, Alerts, and Notifications

> **DataMindAI | Ahmed**

## Making pipelines production-ready

| Feature | Use case |
|---------|----------|
| **Cron schedule** | Run at fixed times (e.g. 9:30am daily) |
| **Continuous** | Always-on streaming jobs |
| **File Arrival trigger** | Fire when a file lands in ADLS/S3 |
| **SQL Alerts** | Email when a metric exceeds a threshold |
| **Job notifications** | Email on success/failure/duration |

## Duration thresholds
| Threshold | Action |
|-----------|--------|
| Expected completion | Webhook fires if exceeded (warning) |
| Maximum completion | Job is KILLED if exceeded (hard stop) |

---
## Scheduling types — reference
---

### Cron schedule examples
Enter these in **Job → Schedules & Triggers → Scheduled → Custom CRON**

| Expression | Meaning |
|-----------|---------|
| `0 9 30 * * ?` | Every day at 9:30am UTC |
| `0 0 6 * * ?` | Every day at 6:00am UTC |
| `0 0 */4 * * ?` | Every 4 hours |
| `0 30 8 ? * MON-FRI` | Weekdays at 8:30am |
| `0 0 23 ? * SAT,SUN` | Weekends at 11pm |
| `0 0 0 1 * ?` | First of every month |

### File Arrival trigger
**Job → Schedules & Triggers → File Arrival**
- Monitor path: `abfss://container@storage.dfs.core.windows.net/incoming/`
- Debounce interval: 60 seconds (prevents duplicate fires)

### Continuous trigger
- Ensures exactly one active run at all times
- Uses exponential backoff if the run fails
- ⚠️ Requires Job Cluster (not Serverless)

---
## SQL Alert — revenue threshold example
---

In [ ]:
# ── Create the data that drives the SQL alert ─────────────────────────────────
# In the Databricks UI: SQL → Alerts → New Alert
# Query: SELECT SUM(total_revenue) AS total FROM demo_catalog.gold.daily_revenue
# Condition: total > 300
# Action: Email on_call@university.ac.uk

from pyspark.sql import Row
import datetime, random

# Simulate daily revenue data for the alert demo
daily = [
    Row(report_date='2024-01-15', course='Data Science', total_revenue=245.0, student_count=4),
    Row(report_date='2024-01-15', course='Engineering',  total_revenue=185.0, student_count=3),
    Row(report_date='2024-01-15', course='Mathematics',  total_revenue=220.0, student_count=3),
]

df = spark.createDataFrame(daily)
df.write.format('delta').mode('overwrite').saveAsTable('demo_catalog.gold.daily_revenue')

total = df.agg({'total_revenue': 'sum'}).collect()[0][0]
threshold = 300

print('=== SQL ALERT SIMULATION ===')
print(f'Total revenue today: £{total:,.2f}')
print(f'Alert threshold:     £{threshold:,.2f}')
print()
if total > threshold:
    print(f'ALERT TRIGGERED — Revenue £{total:,.2f} exceeds threshold £{threshold:,.2f}')
    print('  Action: Email sent to on_call@university.ac.uk')
else:
    print(f'No alert — Revenue £{total:,.2f} is within threshold')
print('===========================')
print()
print('In Databricks UI: SQL → Alerts → New Alert')
print('  Query:     SELECT SUM(total_revenue) AS total FROM demo_catalog.gold.daily_revenue')
print('  Condition: total > 300')
print('  Action:    Email on_call@university.ac.uk')

---
## Job Notifications — configuration guide
---

In [ ]:
# ── Simulate a slow-running job to demo duration alerts ───────────────────────
import time, datetime

# Set these in Job UI:
# Job → Edit → Duration
#   Expected completion: 30 seconds  → webhook fires if exceeded
#   Maximum completion:  5 minutes   → job is killed if exceeded

start = datetime.datetime.now()
print(f'Job started at: {start.strftime("%H:%M:%S")}')
print(f'Expected completion: 30 seconds')
print(f'Maximum completion:  5 minutes')
print()

# Simulate normal work
for i in range(5):
    print(f'Processing step {i+1}/5...')
    time.sleep(3)

# ─── Uncomment to trigger Expected Completion alert ───
# print('Simulating slow run...')
# time.sleep(60)  # 60s — exceeds 30s expected threshold → webhook fires

elapsed = (datetime.datetime.now() - start).seconds
print(f'Job completed in {elapsed} seconds.')
print()
print('Duration alert config in Job UI:')
print('  Job → Edit → scroll to Duration section')
print('  Expected completion: enter 30')
print('  Maximum completion:  enter 300')
print('  Webhook URL: paste Slack/Teams/PagerDuty webhook here')

---
## Full notification config summary
---

In [ ]:
# ── Full notification setup reference ────────────────────────────────────────
print('=== NOTIFICATION CONFIGURATION REFERENCE ===')
print()
print('1. JOB-LEVEL NOTIFICATIONS')
print('   Job → Edit → Notifications → + Add')
print('   Events available:')
print('     on_failure     → email on any task failure')
print('     on_success     → email on full run success')
print('     on_start       → email when job begins')
print('     on_duration_warning_threshold_exceeded → when Expected Completion exceeded')
print()
print('2. TASK-LEVEL NOTIFICATIONS')
print('   Task → Edit → Notifications')
print('   Use to alert on a specific task, not the whole job')
print()
print('3. SLACK INTEGRATION')
print('   Destination → + Add Destination → Webhook')
print('   Paste your Slack Incoming Webhook URL')
print()
print('4. EMAIL GROUPS')
print('   Use a distribution list email to notify multiple people')
print('   e.g. data-ops@university.ac.uk')
print()
print('5. PAGERDUTY')
print('   Use PagerDuty Events API URL as the webhook destination')
print('   For on-call escalation on critical failures')

---
## End-to-end production checklist
---

## Production readiness checklist

Use this before going live with any Lakeflow Job:

| Item | Done? |
|------|-------|
| Schedule configured (cron / file arrival / continuous) | ☐ |
| on_failure notification → ops team email | ☐ |
| on_duration_warning_threshold_exceeded webhook | ☐ |
| Maximum completion (hard timeout) set | ☐ |
| Retry policy on ingestion tasks (1 retry, 15s wait) | ☐ |
| Error handler task (ALL_FAILED dependency) wired up | ☐ |
| Data quality task runs in parallel with transform | ☐ |
| SQL alert on key business metric | ☐ |
| Partial Repair tested (force fail + repair) | ☐ |
| Asset Bundle (YAML) committed to Git repo | ☐ |

In [ ]:
print('Series complete!')
print()
print('Notebooks in this series:')
for i, title in enumerate([
    'Introduction to Lakeflow Jobs',
    'Jobs vs Pipelines',
    'Setup and Prerequisites',
    'Basic Tasks and DAGs',
    'Conditionals (If/Else)',
    'For-Each Loops',
    'Dynamic Task Values',
    'SQL Integration',
    'Advanced: Dynamic Data Ingestion',
    'Scheduling, Alerts, and Notifications',
], 1):
    print(f'  {i:02d}. {title}')
print()
print('GitHub: push all .ipynb files and clone into Databricks Repos')
print('YouTube: DataMindAI | Ahmed')